In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('dirty_model.csv')
df.head()

,First_pokemon,Second_pokemon,Winner,Name_P1,Type 1_P1,Type 2_P1,Type_P1,Abilities_P1,HiddenAbility_P1,Generation_P1,...,DamageFromSteel_P2,DamageFromFire_P2,DamageFromWater_P2,DamageFromGrass_P2,DamageFromElectric_P2,DamageFromPsychic_P2,DamageFromIce_P2,DamageFromDragon_P2,DamageFromDark_P2,DamageFromFairy_P2
0,266,298,298,Larvitar,Rock,Ground,"['Rock', 'Ground']",['Guts'],['Sand Veil'],II,...,1.0,2.0,0.5,0.5,0.5,0.0,2.0,1.0,0.5,2.0
1,702,701,701,Virizion,Grass,Fighting,"['Grass', 'Fighting']",['Justified'],[],V,...,2.0,0.5,2.0,2.0,1.0,2.0,1.0,1.0,0.5,2.0
2,151,231,151,Omastar,Rock,Water,"['Rock', 'Water']","['Swift Swim', 'Shell Armor']",['Weak Armor'],I,...,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,657,752,657,Joltik,Bug,Electric,"['Bug', 'Electric']","['Compound Eyes', 'Unnerve']",['Swarm'],V,...,0.5,2.0,1.0,0.5,1.0,0.5,0.5,0.5,2.0,0.5
4,192,134,134,Natu,Psychic,Flying,"['Psychic', 'Flying']","['Synchronize', 'Early Bird']",['Magic Bounce'],II,...,2.0,2.0,1.0,1.0,1.0,0.5,0.5,1.0,2.0,1.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10070 entries, 0 to 10069
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   First_pokemon          10070 non-null  int64  
 1   Second_pokemon         10070 non-null  int64  
 2   Winner                 10070 non-null  int64  
 3   Name_P1                10070 non-null  object 
 4   Type 1_P1              10070 non-null  object 
 5   Type 2_P1              10070 non-null  object 
 6   Type_P1                10070 non-null  object 
 7   Abilities_P1           10070 non-null  object 
 8   HiddenAbility_P1       10070 non-null  object 
 9   Generation_P1          10070 non-null  object 
 10  Hp_P1                  10070 non-null  float64
 11  Attack_P1              10070 non-null  float64
 12  Defense_P1             10070 non-null  float64
 13  SpecialAttack_P1       10070 non-null  float64
 14  SpecialDefense_P1      10070 non-null  float64
 15  Sp

In [4]:
df['Winner'] = np.where(df['Winner'] == df['First_pokemon'], 0, 1)

In [5]:
obj_cols = df.select_dtypes(include=['object']).columns
obj_cols
df = df.drop(columns=obj_cols)

In [6]:
X = df.drop(columns=['First_pokemon', 'Second_pokemon', 'Winner'])
y = df['Winner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

## **Random Forest**

In [7]:
rf_estimator = RandomForestClassifier(n_estimators=100, random_state=42)
rfe_selector = RFE(estimator=rf_estimator, n_features_to_select=20, step=1)

print("Đang thực hiện Feature Selection...")
rfe_selector.fit(X_train, y_train)

selected_features = X.columns[rfe_selector.support_]
print(f"Các feature được chọn ({len(selected_features)}): {list(selected_features)}")

X_train_selected = rfe_selector.transform(X_train)
X_val_selected = rfe_selector.transform(X_val)
X_test_selected = rfe_selector.transform(X_test)

Đang thực hiện Feature Selection...
Các feature được chọn (20): ['Hp_P1', 'Attack_P1', 'Defense_P1', 'SpecialAttack_P1', 'SpecialDefense_P1', 'Speed_P1', 'TotalStats_P1', 'Weight_P1', 'Height_P1', 'CatchRate_P1', 'Hp_P2', 'Attack_P2', 'Defense_P2', 'SpecialAttack_P2', 'SpecialDefense_P2', 'Speed_P2', 'TotalStats_P2', 'Weight_P2', 'Height_P2', 'CatchRate_P2']


In [8]:
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_final.fit(X_train_selected, y_train) 

train_acc = accuracy_score(y_train, rf_final.predict(X_train_selected))
val_acc = accuracy_score(y_val, rf_final.predict(X_val_selected))
test_acc = accuracy_score(y_test, rf_final.predict(X_test_selected))

print("=== Random Forest ===") 

print(f"Train: {train_acc:.4f}")
print(f"Val: {val_acc:.4f}")
print(f"Test: {test_acc:.4f}")

=== Random Forest ===
Train: 1.0000
Val: 0.9256
Test: 0.9166


## **Gradient Boosting**

In [9]:
gb_estimator = GradientBoostingClassifier(n_estimators=100, random_state=42)

rfe_selector = RFE(estimator=gb_estimator, n_features_to_select=20, step=1)

print("Đang thực hiện Feature Selection...")

rfe_selector.fit(X_train, y_train) 

selected_features = X.columns[rfe_selector.support_]
print(f"Các feature được chọn ({len(selected_features)}): {list(selected_features)}")

X_train_selected = rfe_selector.transform(X_train)
X_val_selected = rfe_selector.transform(X_val)
X_test_selected = rfe_selector.transform(X_test)

Đang thực hiện Feature Selection...
Các feature được chọn (20): ['Hp_P1', 'Attack_P1', 'Defense_P1', 'Speed_P1', 'TotalStats_P1', 'Weight_P1', 'Height_P1', 'DamageFromGround_P1', 'DamageFromPsychic_P1', 'Hp_P2', 'Attack_P2', 'Defense_P2', 'Speed_P2', 'TotalStats_P2', 'Weight_P2', 'CatchRate_P2', 'DamageFromPoison_P2', 'DamageFromElectric_P2', 'DamageFromPsychic_P2', 'DamageFromDark_P2']


In [10]:
final_model = GradientBoostingClassifier(
    n_estimators=200, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)

final_model.fit(X_train_selected, y_train) 

train_acc = accuracy_score(y_train, final_model.predict(X_train_selected))
val_acc = accuracy_score(y_val, final_model.predict(X_val_selected))
test_acc = accuracy_score(y_test, final_model.predict(X_test_selected))

print("=== Gradient Boosting ===") 

print(f"Train: {train_acc:.4f}")
print(f"Val: {val_acc:.4f}")
print(f"Test: {test_acc:.4f}")

=== Gradient Boosting ===
Train: 0.9352
Val: 0.9156
Test: 0.9131


## **Logistic Regression**

In [11]:
log_estimator = LogisticRegression(max_iter=1000, n_jobs=-1)

rfe_log = RFE(
    estimator=log_estimator,
    n_features_to_select=20,   # hoặc 10, tuỳ bạn muốn
    step=1
)

print("Đang thực hiện Feature Selection (Logistic Regression)...")
rfe_log.fit(X_train, y_train)

log_selected_features = X.columns[rfe_log.support_]
print(f"Các feature LogReg chọn ({len(log_selected_features)}): {list(log_selected_features)}")

X_train_log = rfe_log.transform(X_train)
X_val_log   = rfe_log.transform(X_val)
X_test_log  = rfe_log.transform(X_test)


Đang thực hiện Feature Selection (Logistic Regression)...
Các feature LogReg chọn (20): ['IsLegendary_P1', 'IsMythical_P1', 'HasMega_P1', 'EvoStage_P1', 'TotalEvoStages_P1', 'DamageFromNormal_P1', 'DamageFromFlying_P1', 'DamageFromRock_P1', 'DamageFromSteel_P1', 'DamageFromGrass_P1', 'DamageFromIce_P1', 'DamageFromDark_P1', 'DamageFromFairy_P1', 'IsMythical_P2', 'EvoStage_P2', 'TotalEvoStages_P2', 'DamageFromNormal_P2', 'DamageFromFlying_P2', 'DamageFromGround_P2', 'DamageFromSteel_P2']


In [12]:
logreg_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, n_jobs=-1))
])

logreg_clf.fit(X_train_log, y_train)

print("=== Logistic Regression ===")
for name, X_split, y_split in [
    ("Train", X_train_log, y_train),
    ("Val",   X_val_log,   y_val),
    ("Test",  X_test_log,  y_test),
]:
    y_pred = logreg_clf.predict(X_split)
    print(f"{name} accuracy: {accuracy_score(y_split, y_pred):.4f}")


=== Logistic Regression ===
Train accuracy: 0.7021
Val accuracy: 0.6964
Test accuracy: 0.6986


## **Decision Tree**

In [13]:
dt_estimator = DecisionTreeClassifier(
    max_depth=None,      # có thể tune
    random_state=42
)

rfe_dt = RFE(
    estimator=dt_estimator,
    n_features_to_select=20,   # cùng số lượng để so sánh cho công bằng
    step=1
)

print("Đang thực hiện Feature Selection (Decision Tree)...")
rfe_dt.fit(X_train, y_train)

dt_selected_features = X.columns[rfe_dt.support_]
print(f"Các feature Decision Tree chọn ({len(dt_selected_features)}): {list(dt_selected_features)}")

X_train_dt = rfe_dt.transform(X_train)
X_val_dt   = rfe_dt.transform(X_val)
X_test_dt  = rfe_dt.transform(X_test)


Đang thực hiện Feature Selection (Decision Tree)...
Các feature Decision Tree chọn (20): ['Hp_P1', 'Attack_P1', 'SpecialAttack_P1', 'SpecialDefense_P1', 'Speed_P1', 'TotalStats_P1', 'Weight_P1', 'DamageFromGround_P1', 'Hp_P2', 'Attack_P2', 'Defense_P2', 'SpecialAttack_P2', 'SpecialDefense_P2', 'Speed_P2', 'TotalStats_P2', 'Weight_P2', 'Height_P2', 'DamageFromFighting_P2', 'DamageFromFire_P2', 'DamageFromElectric_P2']


In [14]:
dt_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=8, min_samples_split=10, random_state=42))
])

dt_clf.fit(X_train_dt, y_train)

print("=== Decision Tree  ===")
for name, X_split, y_split in [
    ("Train", X_train_dt, y_train),
    ("Val",   X_val_dt,   y_val),
    ("Test",  X_test_dt,  y_test),
]:
    y_pred = dt_clf.predict(X_split)
    print(f"{name} accuracy: {accuracy_score(y_split, y_pred):.4f}")


=== Decision Tree  ===
Train accuracy: 0.9477
Val accuracy: 0.9173
Test accuracy: 0.9096
